# 🎯 Surveillance AI - Streamlined Colab Notebook

This notebook runs video analysis directly in Colab, using OpenAI or Gemini, and Google Video Intelligence. All results and uploads are saved to your Google Drive. No server, no agents, no endless loops.

**Features:**
- Mounts Google Drive, auto-creates `/uploads` and `/results` folders
- Loads `credentials.json` and `.env` from Drive
- Supports OpenAI (default) and Gemini (switchable)
- Manual video uploads only
- Outputs JSON analysis and video previews to Drive
- Minimal manual steps, clear error handling


In [ ]:
# Step 1: Mount Google Drive and Setup Project Folders
from google.colab import drive
import os
drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/surveillance-ai'
uploads_path = os.path.join(project_path, 'uploads')
results_path = os.path.join(project_path, 'results')
os.makedirs(uploads_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)
print(f'✅ Project directory: {project_path}')
print(f'✅ Uploads directory: {uploads_path}')
print(f'✅ Results directory: {results_path}')


In [ ]:
# Step 2: Load Environment Variables and Google Credentials
from dotenv import load_dotenv
import os
env_path = os.path.join(project_path, '.env')
credentials_path = os.path.join(project_path, 'credentials.json')
load_dotenv(env_path)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credentials_path

openai_key = os.getenv('OPENAI_API_KEY')
gemini_key = os.getenv('GEMINI_API_KEY')

print(f'📄 .env file exists: {os.path.exists(env_path)}')
print(f'🔑 OpenAI API Key loaded: {'✅' if openai_key else '❌'}')
print(f'🔑 Gemini API Key loaded: {'✅' if gemini_key else '❌'}')
print(f'🔑 Google Cloud credentials: {'✅' if os.path.exists(credentials_path) else '❌'}')
if not openai_key and not gemini_key:
    print('⚠️  Warning: No API keys found in .env!')


In [ ]:
# Step 3: Install Required Packages
!pip install --quiet openai google-cloud-videointelligence google-generativeai python-dotenv opencv-python tqdm


## 📤 Step 4: Upload a Video to Analyze
Upload your video to the `/uploads` folder in your Google Drive (`surveillance-ai/uploads`). Refresh the folder in Drive if needed.


In [ ]:
# Step 5: List Uploaded Videos
import glob
uploaded_videos = glob.glob(os.path.join(uploads_path, '*.mp4'))
print('Uploaded videos:')
for v in uploaded_videos:
    print('-', os.path.basename(v))
if not uploaded_videos:
    print('No videos found. Please upload to /uploads in Drive.')


## 🧠 Step 6: Choose Model (OpenAI or Gemini)
Set `use_gemini = True` to use Gemini, otherwise OpenAI will be used.


In [ ]:
# Set this flag to switch between OpenAI and Gemini
use_gemini = False  # Set True to use Gemini, False for OpenAI


## 🎬 Step 7: Analyze a Video
Call the function below to analyze a video. Results will be saved to `/results` in Drive.


In [ ]:
import json
from tqdm import tqdm
import cv2
from google.cloud import videointelligence_v1 as vi

def analyze_video(video_path, prompt, use_gemini=False):
    # 1. Google Video Intelligence API
    client = vi.VideoIntelligenceServiceClient()
    with open(video_path, 'rb') as f:
        input_content = f.read()
    features = [vi.Feature.LABEL_DETECTION, vi.Feature.EXPLICIT_CONTENT_DETECTION]
    operation = client.annotate_video({
        'features': features,
        'input_content': input_content,
    })
    print('⏳ Waiting for Google Video Intelligence API...')
    result = operation.result(timeout=300)
    annotation = result.annotation_results[0]
    # 2. Extract labels and explicit content
    labels = [l.entity.description for l in annotation.segment_label_annotations]
    explicit = getattr(annotation, 'explicit_annotation', None)
    explicit_frames = []
    if explicit and hasattr(explicit, 'frames'):
        for frame in explicit.frames:
            explicit_frames.append({
                'time_offset': frame.time_offset.total_seconds(),
                'pornography_likelihood': vi.Likelihood(frame.pornography_likelihood).name
            })
    # 3. Frame extraction (preview)
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    preview_frames = []
    for i in tqdm(range(0, frame_count, max(1, frame_count // 5)), desc='Extracting preview frames'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            preview_path = os.path.join(results_path, f'preview_{os.path.basename(video_path)}_{i}.jpg')
            cv2.imwrite(preview_path, frame)
            preview_frames.append(preview_path)
    cap.release()
    # 4. LLM analysis
    summary = ''
    if use_gemini:
        import google.generativeai as genai
        genai.configure(api_key=gemini_key)
        model = genai.GenerativeModel('gemini-pro')
        response = model.generate_content(f'Video labels: {labels}. Prompt: {prompt}')
        summary = response.text
    else:
        import openai
        openai.api_key = openai_key
        completion = openai.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[
                {"role": "system", "content": "You are a helpful video analysis assistant."},
                {"role": "user", "content": f'Video labels: {labels}. Prompt: {prompt}'}
            ]
        )
        summary = completion.choices[0].message.content
    # 5. Save results
    result_json = {
        'video': os.path.basename(video_path),
        'labels': labels,
        'explicit_content': explicit_frames,
        'llm_summary': summary,
        'preview_frames': preview_frames
    }
    out_path = os.path.join(results_path, os.path.basename(video_path) + '_analysis.json')
    with open(out_path, 'w') as f:
        json.dump(result_json, f, indent=2)
    print(f'✅ Analysis complete! Results saved to: {out_path}')
    return result_json


## ▶️ Step 8: Run Analysis
Example usage (edit as needed):
```python
video_path = os.path.join(uploads_path, 'your_video.mp4')
prompt = 'Detect any suspicious activity'
result = analyze_video(video_path, prompt, use_gemini=use_gemini)
print(json.dumps(result, indent=2))
```
